# Week 3 · Module 4 Practical — Connect & Delegate 🔌

**PMS RoBoTics Research Center · 4-Week Internship & Training Program**
**Week 3 · Module 4: MCP & multi-agent systems**

---

| # | Rung | You build |
|---|------|-----------|
| 1 | An MCP-style server | Tools behind a `discover()` / `call()` boundary |
| 2 | An MCP-style client | Discovers tools at runtime, feeds them to the loop (slide 5, live) |
| 3 | Agent over the server | The Day 3 loop, but tools come from discovery — nothing hardcoded |
| 4 | Specialist agents | A sales agent and an orders agent, each with few tools |
| 5 | **The supervisor** | An agent whose tools ARE other agents (slide 11, live) |
| 🏁 | **Route real questions** | Mixed queries → the right specialist, delegation trace printed |

> 🧭 Slide 5 (MCP sequence) and slide 11 (supervisor sequence) are the map for today — the code prints the same steps.
> 📎 We build MCP *in-process* to stay dependency-light and see every wire. A real `stdio` MCP note is in the appendix.

## Step 0 — Setup 🔑

In [ ]:
%pip install -q -U openai

In [ ]:
import json
from getpass import getpass
from openai import OpenAI

API_KEY = getpass("Paste the OpenAI API key: ")
client = OpenAI(api_key=API_KEY)
MODEL = "gpt-4o-mini"

# SmartMart "live" systems
INVENTORY = {"P-101": 42, "P-103": 65, "P-105": 23, "P-118": 3, "P-119": 14}
PRICES    = {"P-101": 1299, "P-103": 899, "P-105": 3499, "P-118": 12999, "P-119": 2799}
NAMES     = {"P-101": "Prima Electric Kettle 1.5L", "P-103": "Zenith Steel Kettle 1.0L",
             "P-105": "SwiftMix Mixer Grinder 750W", "P-118": "CleanSweep Robot Vacuum",
             "P-119": "FreshBrew Coffee Maker 600ml"}
ORDERS    = {"4521": {"status": "shipped", "eta": "tomorrow", "item": "P-105"},
             "7788": {"status": "processing", "eta": "3 days", "item": "P-101"}}
print("✅ Ready.")

---
## Rung 1 — An MCP-style server 🖥️

The essence of MCP, stripped to two verbs: **the server exposes tools you can DISCOVER and CALL** — without importing its code. In real MCP these travel as JSON-RPC over stdio/HTTP; here they cross a class boundary, but the SHAPE is identical.

In [ ]:
class MCPServer:
    """A minimal MCP-style tool server. Owns its tools; exposes discover() + call()."""

    def __init__(self, name):
        self.name = name
        self._tools = {}       # name -> (func, schema)

    def tool(self):
        """Decorator — like @mcp.tool(). Reads the docstring to build the schema."""
        def wrap(func):
            import inspect
            params = {p: {"type": "string"} for p in inspect.signature(func).parameters}
            self._tools[func.__name__] = (func, {
                "type": "function",
                "function": {"name": func.__name__,
                             "description": (func.__doc__ or "").strip(),
                             "parameters": {"type": "object", "properties": params,
                                            "required": list(params)}}})
            return func
        return wrap

    def discover(self):                       # ← MCP 'tools/list'
        return [schema for _, schema in self._tools.values()]

    def call(self, name, args):               # ← MCP 'tools/call'
        if name not in self._tools:
            return f"Unknown tool '{name}' on server '{self.name}'"
        return str(self._tools[name][0](**args))

# The catalog team ships this server — you never see their DB code:
catalog_server = MCPServer("catalog")

@catalog_server.tool()
def get_price(product_id: str) -> str:
    """The current price in rupees of a product, given its catalog ID like P-101."""
    return f"{NAMES.get(product_id,'?')}: Rs. {PRICES.get(product_id,'unknown'):,}" if product_id in PRICES else "unknown ID"

@catalog_server.tool()
def check_inventory(product_id: str) -> str:
    """How many units of a product are in stock right now, given its catalog ID."""
    return f"{INVENTORY[product_id]} units of {NAMES[product_id]}" if product_id in INVENTORY else "unknown ID"

@catalog_server.tool()
def list_products(keyword: str) -> str:
    """Search the catalog by keyword; returns matching IDs, names and prices."""
    hits = [f"{p}: {NAMES[p]} (Rs. {PRICES[p]:,})" for p in NAMES if keyword.lower() in NAMES[p].lower()]
    return "\n".join(hits) or f"nothing matching '{keyword}'"

print("catalog server exposes:", [t["function"]["name"] for t in catalog_server.discover()])

---
## Rung 2 — An MCP-style client + the discovery diagram 🔎

Now the CLIENT side (slide 5). The agent doesn't import the server's functions — it **discovers** them at runtime, then **calls** them by name. Watch the four discovery steps print:

In [ ]:
def mcp_agent(question, servers, max_steps=5, verbose=True):
    """An agent whose tools are DISCOVERED from MCP servers — nothing hardcoded."""
    # ---- DISCOVERY (slide 5, steps 1-4): happens once, before the loop ----
    schema, route = [], {}          # route: tool_name -> which server owns it
    for srv in servers:
        for t in srv.discover():
            schema.append(t)
            route[t["function"]["name"]] = srv
    if verbose:
        print(f"🔌 discovered {len(schema)} tools from {len(servers)} server(s): "
              f"{list(route)}")

    msgs = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=schema)
        m = r.choices[0].message
        if not m.tool_calls:
            return m.content
        msgs.append(m)
        for tc in m.tool_calls:                       # ---- CALL (steps 5-8) ----
            srv = route[tc.function.name]              # which server owns this tool?
            args = json.loads(tc.function.arguments)
            result = srv.call(tc.function.name, args)  # relay across the boundary
            if verbose: print(f"   → {srv.name}.{tc.function.name}({args}) = {result[:50]}")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "Step limit reached."

print("🛒", mcp_agent("What's the cheapest kettle and is it in stock?", [catalog_server]))

**What just happened, mapped to slide 5:** the agent asked each server what it offers (`discover()`), registered the schemas, then the LLM loop called tools *by name* — the client routed each call to the owning server. **The agent never imported `get_price`.** Swap in a different catalog server tomorrow and the agent code doesn't change.

**✏️ Your turn:** stand up a SECOND server (`orders_server`) with a `track_order(order_id)` tool, pass `[catalog_server, orders_server]`, and ask a question that needs both. One agent, two servers, zero rewiring.

In [ ]:
# A second server — the orders team owns this one:
orders_server = MCPServer("orders")

@orders_server.tool()
def track_order(order_id: str) -> str:
    """Status and delivery ETA of a customer order, given the order number like 4521."""
    o = ORDERS.get(order_id)
    return f"Order {order_id} ({NAMES[o['item']]}): {o['status']}, ETA {o['eta']}" if o else f"no order {order_id}"

# One agent, discovering from BOTH servers:
print("🛒", mcp_agent("Is order 4521 shipped, and what did that product cost?",
                     [catalog_server, orders_server]))

---
## Rung 3–4 — Specialist agents 🧑‍🔧

Slide 8's idea: instead of one agent juggling every tool, build **small specialists**. Each is just the Day 3 loop with a focused prompt + a subset of servers. We wrap the loop in a function so it can become a *tool* in Rung 5:

In [ ]:
def run_agent(question, servers, system, max_steps=4):
    """A reusable specialist: focused system prompt + its own servers. Returns a string."""
    schema, route = [], {}
    for srv in servers:
        for t in srv.discover():
            schema.append(t); route[t["function"]["name"]] = srv
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": question}]
    for _ in range(max_steps):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=schema)
        m = r.choices[0].message
        if not m.tool_calls:
            return m.content
        msgs.append(m)
        for tc in m.tool_calls:
            result = route[tc.function.name].call(tc.function.name, json.loads(tc.function.arguments))
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "…(specialist hit its step limit)"

def sales_agent(question: str) -> str:
    """Handles product questions: prices, stock, catalog search, recommendations."""
    return run_agent(question, [catalog_server],
        "You are SmartMart's sales specialist. Use the catalog tools to answer product, "
        "price, stock and recommendation questions in <=3 sentences.")

def orders_agent(question: str) -> str:
    """Handles order questions: status, delivery ETA, tracking."""
    return run_agent(question, [orders_server],
        "You are SmartMart's orders specialist. Use the order tools to answer status and "
        "delivery questions in <=3 sentences.")

print("sales  :", sales_agent("what's your cheapest kettle?"))
print("orders :", orders_agent("where is order 7788?"))

**Each specialist is testable in isolation** — short prompt, few tools, one job. That's the whole benefit of splitting: you can prompt, debug and evaluate each one on its own (Week 2's eval skills apply per-agent).

**Now the twist that makes it multi-agent:** notice `sales_agent` and `orders_agent` are plain functions with docstrings. Anything with a docstring can be… a tool. 👀

---
## Rung 5 — The supervisor 👔

Slide 9 & 11: an agent whose **tools are other agents**. `sales_agent` and `orders_agent` become the supervisor's registry — Day 1's loop, recursed. Watch the delegation trace and match it to slide 11:

In [ ]:
SPECIALISTS = {"sales_agent": sales_agent, "orders_agent": orders_agent}
SPECIALIST_SCHEMA = [{
    "type": "function",
    "function": {"name": name, "description": fn.__doc__.strip(),
                 "parameters": {"type": "object",
                                "properties": {"question": {"type": "string",
                                    "description": "the user's question, passed through"}},
                                "required": ["question"]}}}
    for name, fn in SPECIALISTS.items()]

def supervisor(question, max_steps=5, verbose=True):
    """Routes to specialist agents, then synthesizes. Its 'tools' are whole agents."""
    msgs = [{"role": "system", "content":
             "You are SmartMart's support supervisor. Delegate each question to the right "
             "specialist agent, then give the customer a single clear answer. If a question "
             "spans both, call both."},
            {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        r = client.chat.completions.create(model=MODEL, messages=msgs, tools=SPECIALIST_SCHEMA)
        m = r.choices[0].message
        if not m.tool_calls:
            if verbose: print(f"  step {step} · SUPERVISOR answers")
            return m.content
        if m.content and verbose:
            print(f"  step {step} · THINK → {m.content}")
        msgs.append(m)
        for tc in m.tool_calls:
            args = json.loads(tc.function.arguments)
            if verbose: print(f"  step {step} · DELEGATE → {tc.function.name}({args['question'][:40]}…)")
            result = SPECIALISTS[tc.function.name](**args)     # a whole agent runs here!
            if verbose: print(f"  step {step} · ⤶ specialist returned: {result[:55]}…")
            msgs.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    return "Step limit reached."

print("🛒", supervisor("where is my order 4521, and how much did that item cost?"))

**Read the trace against slide 11:** the supervisor THINKs (this is an orders + sales job), DELEGATEs to `orders_agent` (which runs its *own* loop and its *own* tool calls — loops inside loops), gets the result back, delegates to `sales_agent`, then synthesizes one answer. That nesting — an agent-as-tool running a nested loop — is the entire idea of multi-agent systems.

**✏️ Your turn:** ask a pure-sales question and a pure-orders question. Confirm the supervisor delegates to exactly ONE specialist each time — no wasted agent calls. (Over-delegation is the #1 multi-agent cost leak.)

---
## 🏁 Finale — route mixed traffic 🚦

Five real customer messages through the supervisor. Count the delegations — that's your cost:

In [ ]:
queries = [
    "how much is the robot vacuum?",                       # sales only
    "track order 7788 please",                             # orders only
    "is the mixer grinder in stock and where's order 4521?",  # both
    "recommend something for making coffee under 3000",    # sales only
    "what's the weather in Pune?",                         # neither — watch it refuse
]
for q in queries:
    print(f"🧑 {q}")
    print(f"🛒 {supervisor(q, verbose=False)}")
    print("-" * 72)

In [ ]:
# 💬 Talk to the SmartMart supervisor — 'quit' to stop, 'v' prefix for the delegation trace.
while True:
    user_msg = input("You: ")
    if user_msg.strip().lower() in ("quit", "exit", "q"):
        print("👋 Tomorrow: assemble the graded SmartMart agent.")
        break
    v = user_msg.startswith("v ")
    print("🛒", supervisor(user_msg[2:] if v else user_msg, verbose=v))

---
## 📎 Appendix — a REAL MCP server (read, optional to run)

Our in-process version showed the *shape*. A real MCP server runs as a separate process and speaks JSON-RPC. Here's the same `get_price` as a genuine stdio server — this is what you'd publish for others to plug in:

```python
# save as smartmart_server.py, then: pip install "mcp[cli]"
from mcp.server.fastmcp import FastMCP
mcp = FastMCP("smartmart")

@mcp.tool()
def get_price(product_id: str) -> str:
    """The current price in rupees of a product, given its catalog ID."""
    return f"Rs. {PRICES.get(product_id, 'unknown')}"

if __name__ == "__main__":
    mcp.run()          # speaks JSON-RPC over stdio
```

```python
# a real client discovers + calls it over stdio:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import asyncio

async def main():
    params = StdioServerParameters(command="python", args=["smartmart_server.py"])
    async with stdio_client(params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()               # ← discover()
            out = await session.call_tool("get_price", {"product_id": "P-101"})  # ← call()
            print([t.name for t in tools.tools], "→", out.content[0].text)

asyncio.run(main())
```

**The mapping is exact:** `list_tools` = our `discover()`, `call_tool` = our `call()`, JSON-RPC/stdio = our class boundary. You already understand real MCP — it's the in-process version with a socket in the middle. (Colab's async + subprocess handling can be fiddly, which is why the lesson used in-process; try the real thing on your own machine.)

---
## 🏆 Homework (20 min, choose your Friday architecture)

1. **Decide: single or supervisor** — for the SmartMart project (inventory + orders + pricing), which will you build? Write 3 sentences: how many tools, is one agent's prompt overflowing, does the extra delegation cost pay for itself? (There's no wrong answer — a defended single agent is often the right call.)
2. **Third server** — add a `policy_server` (returns/delivery/warranty tools) and a `support_agent` specialist, register it with the supervisor, and prove a policy question routes correctly. Now you have the full three-specialist shape from slide 8.
3. **Cost audit** — run 5 mixed queries through the supervisor and count total LLM calls (supervisor + every specialist's internal loop). Compare with a single agent holding all the tools answering the same 5. Which is cheaper here? (Small corpora often favour the single agent — this is exactly Friday's design decision, priced.)

### What's next — assembly day 🏗️
**Day 5 — Week 3 Mini-Project (graded):** the SmartMart agent that checks inventory, tracks orders, and looks up pricing — your architecture, your tools, your traces. Everything from Days 1–4 in one deliverable.

---
*PMS RoBoTics Research Center · pmsroboticsrc@gmail.com · (+91) 9960664674*